# Backend Notebook Wrapper
Notebook nay chi dong vai tro wrapper de goi backend module moi trong `back-end/app`.

## 1) Cai dat dependencies
Cai package tu file `requirements.txt` cua backend.

In [ ]:
%pip install -r requirements.txt -q

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached triton-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)


## 2) Nap bien moi truong
Nap `.env` va (neu dang o Colab) map key tu `userdata` vao environment.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Ho tro Colab: doc key tu userdata neu co
try:
    from google.colab import userdata  # type: ignore
    for key in ["GOOGLE_API_KEY", "QDRANT_URL", "QDRANT_API_KEY", "HUGGINGFACE_API_KEY", "QDRANT_COLLECTION_NAME", "EMBEDDING_MODEL_NAME", "GENERATE_MODEL_NAME", "NGROK_TOKEN", "NGROK_STATIC_DOMAIN"]:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
except Exception:
    pass

print("Env loaded. GOOGLE_API_KEY set:", bool(os.getenv("GOOGLE_API_KEY")))
print("QDRANT_URL set:", bool(os.getenv("QDRANT_URL")))
print("QDRANT_COLLECTION_NAME:", os.getenv("QDRANT_COLLECTION_NAME", "ITUS_e5_600v2"))

In [ ]:
import sys

# Dam bao co the import tu thu muc back-end
backend_dir = Path.cwd().resolve()
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from app import create_app
from app.config import load_configs

backend_config, pipeline_config = load_configs()
app_api = create_app()

print("Backend module loaded.")
print("Collection:", pipeline_config.collection_name)
print("Embedding model:", pipeline_config.embedding_model_name)
print("Generate model:", backend_config.generate_model_name)

## 3) Chay API server
Khoi dong FastAPI app tu module moi.

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run(app_api, host="0.0.0.0", port=8000)

In [ ]:
## 4) (Optional) Expose bang ngrok
#Chi chay cell nay neu ban can public URL tu Colab/local.

In [ ]:
from pyngrok import ngrok

ngrok_token = os.getenv("NGROK_TOKEN", "")
ngrok_domain = os.getenv("NGROK_STATIC_DOMAIN", "")

if not ngrok_token:
    raise ValueError("Missing NGROK_TOKEN")

ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, domain=ngrok_domain or None)
print("Public URL:", tunnel.public_url)